# Pipeline de Dados Agrícolas: Extração e Análise de Produção de Cana-de-Açúcar (IBGE)

## 📌 Contexto do Projeto
Este projeto faz parte de um pipeline **End-to-End** focado no setor sucroenergético brasileiro. O objetivo é extrair dados oficiais da **Pesquisa Agrícola Municipal (PAM)** através da API SIDRA do IBGE, realizando o tratamento e a modelagem para análise de indicadores de performance agrícola (KPIs).

---

## 🚀 Tecnologias Utilizadas
* **Python**
* **Pandas & Numpy**: Manipulação e limpeza de dados.
* **Requests**: Consumo de API REST.
* **Matplotlib/Seaborn**: Validação visual de dados.

---

## 🛠️ Etapas do Notebook
1. **Extração**: Coleta de dados via API (Cana-de-açúcar em SP | 2018-2023).
2. **Data Cleaning**: Tratamento de valores ausentes (`-`, `...`) e conversão de tipos.
3. **Validação**: Verificação da integridade do dataset e estatísticas básicas.
4. **Modelagem (Próximo Passo)**: Reestruturação para formato analítico (Pivot Table).

---
**Autor:** Agnaldo Gonzaga  
**Setor:** Análise de Controle Agrícola (CTT)

In [1]:
# Standard library
import os

# Manipulação de dados
import numpy as np
import pandas as pd

# Requisição / APIs
import requests
import requests_cache



In [2]:
# URL DA API DO SITE DO IBG COM DADOS DE CANA DE AÇUCAR DE 2018 A 2023
url_api = (
    "https://servicodados.ibge.gov.br/api/v3/agregados/1612/periodos/2018|2019|2020|2021|2022|2023/"
    "variaveis/109|216|214?classificacao=81[31548]&localidades=N3[35]"
)
print(">>> Coletando dados de Cana-de-Açúcar no IBGE...")

>>> Coletando dados de Cana-de-Açúcar no IBGE...


In [3]:
response = requests.get(url_api)
data = response.json()

registros = []

In [4]:

# função para limpar valores do IBGE
def limpar_valor(valor):
    try:
        return float(valor)
    except:
        return np.nan   # valor faltante correto

In [6]:
# ----------------------------
# Processamento dos Dados
# ----------------------------

# Função para limpar valores do IBGE (definida no início do seu notebook)
def limpar_valor(valor):
    try:
        # Tratamento de padrões comuns do IBGE antes da conversão
        if valor in ['-', '...', None] or str(valor).strip() == '':
            return 0.0
        
        #  o formato numérico use ponto como separador decimal
        valor_limpo = str(valor).replace(',', '.')
        return float(valor_limpo)
    except (ValueError, TypeError):
        return np.nan

registros = []

for metrica in data:
    nome_var = metrica['variavel']
    
    for resultado in metrica['resultados']:
        # Pega o nome do produto (Cana-de-açúcar)
        nome_produto = list(resultado['classificacoes'][0]['categoria'].values())[0]
        
        # Percorre as séries de dados (Localidades)
        for serie in resultado['series']:
            localidade = serie['localidade']['nome']
            
            for ano, valor in serie['serie'].items():
                
                # 
                # função criada para processar o valor bruto
                valor_num = limpar_valor(valor)
                
                # Garantia de integridade para Carga SQL (evitando NaNs)
                if np.isnan(valor_num):
                    valor_num = 0.0

                registros.append({
                    'ano': int(ano),
                    'uf': localidade,
                    'produto': nome_produto,
                    'metrica': nome_var,
                    'valor': valor_num
                })

# Criação do DataFrame
df = pd.DataFrame(registros)

# ----------------------------
# Validação dos Dados
# ----------------------------
print("\n--- Informações Gerais do Dataset ---")
print(df.info())

print("\n--- Estatísticas Básicas ---")
print(df['valor'].describe())

print("\n✓ Processamento e limpeza concluídos com sucesso!")


--- Informações Gerais do Dataset ---
<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ano      18 non-null     int64  
 1   uf       18 non-null     str    
 2   produto  18 non-null     str    
 3   metrica  18 non-null     str    
 4   valor    18 non-null     float64
dtypes: float64(1), int64(1), str(3)
memory usage: 852.0 bytes
None

--- Estatísticas Básicas ---
count    18.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: valor, dtype: float64

✓ Processamento e limpeza concluídos com sucesso!
